# NOAA GHCN-D — Verificación de estructura de un CSV de estación

**Objetivo:** inspeccionar un único archivo de estación, sin transformar datos, antes de definir reglas de calidad y cobertura del ETL.

Se descarga automáticamente desde el bucket público NOAA `noaa-ghcn-pds`, en `csv.gz/by_station/`, usando acceso anónimo. Se descomprime a CSV para revisar sus columnas.

**Alcance:** esta prueba confirma el formato CSV de origen de NOAA.


In [1]:
%pip -q install boto3

import csv
import gzip
import io
import shutil
from collections import Counter
from pathlib import Path
import boto3
from botocore import UNSIGNED
from botocore.config import Config

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
codigo_pais = 'NZ'

In [6]:
BUCKET = "noaa-ghcn-pds"
PREFIJO = "csv.gz/by_station/"+codigo_pais

s3 = boto3.client("s3", region_name="us-east-1", config=Config(signature_version=UNSIGNED))
respuesta = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIJO, MaxKeys=1000)
claves = sorted(obj["Key"] for obj in respuesta.get("Contents", []) if obj["Key"].endswith(".csv.gz"))
if not claves:
    raise ValueError("No se encontraron CSV comprimidos de estaciones NZ bajo el prefijo indicado")

# Se elige una estación concreta (la primera por orden alfabético) para hacer la prueba reproducible.
clave = claves[0]
archivo = Path(clave).name.removesuffix(".gz")
print("Estación seleccionada:", archivo.removesuffix(".csv"))
print("Objeto público:", clave)

contenido = s3.get_object(Bucket=BUCKET, Key=clave)["Body"].read()
with gzip.GzipFile(fileobj=io.BytesIO(contenido)) as comprimido:
    with open(archivo, "wb") as salida:
        shutil.copyfileobj(comprimido, salida)
print("CSV descargado y descomprimido:", archivo)
archivo = Path(archivo)

Estación seleccionada: NZ000093012
Objeto público: csv.gz/by_station/NZ000093012.csv.gz
CSV descargado y descomprimido: NZ000093012.csv


## Inspección del formato original

Se leen los campos **como texto**, sin convertir fechas, unidades ni indicadores. Se muestra el primer registro completo y se comprueba si hay encabezado. El conteo de columnas se realiza sobre todo el archivo.

In [7]:
recuento_columnas = Counter()
ejemplos = []
filas_vacias = 0
filas_totales = 0

with archivo.open("r", encoding="utf-8-sig", newline="") as f:
    lector = csv.reader(f)
    for fila in lector:
        if not fila or all(not x.strip() for x in fila):
            filas_vacias += 1
            continue
        filas_totales += 1
        recuento_columnas[len(fila)] += 1
        if len(ejemplos) < 6:
            ejemplos.append(fila)

if not ejemplos:
    raise ValueError("El CSV no contiene registros")

print("Cantidad de filas no vacías (incluye encabezado, si existe):", filas_totales)
print("Filas vacías:", filas_vacias)
print("Cantidad de columnas por fila:", dict(sorted(recuento_columnas.items())))
print("Primera fila (lista completa):", repr(ejemplos[0]))
print("Siguientes filas de ejemplo:")
for fila in ejemplos[1:]:
    print(repr(fila))

primera = [v.strip().upper() for v in ejemplos[0]]
parece_encabezado = ("DATE" in primera or "FECHA" in primera) and ("ELEMENT" in primera or "VARIABLE" in primera)
print("¿La primera fila parece encabezado?:", parece_encabezado)
print("Estructura uniforme:", len(recuento_columnas) == 1)
if parece_encabezado:
    print("Nombres de columnas:", ejemplos[0])
else:
    print("ATENCIÓN: no se detectó un encabezado reconocible; verificar el formato antes de continuar")

Cantidad de filas no vacías (incluye encabezado, si existe): 47998
Filas vacías: 0
Cantidad de columnas por fila: {8: 47998}
Primera fila (lista completa): ['NZ000093012', '19651231', 'TAVG', '194', 'H', '', 'S', '']
Siguientes filas de ejemplo:
['NZ000093012', '19660102', 'TMAX', '233', '', '', 'S', '']
['NZ000093012', '19660104', 'TMAX', '244', '', '', 'S', '']
['NZ000093012', '19660105', 'TMAX', '233', '', '', 'S', '']
['NZ000093012', '19660106', 'TMAX', '233', '', '', 'S', '']
['NZ000093012', '19660107', 'TMAX', '239', '', '', 'S', '']
¿La primera fila parece encabezado?: False
Estructura uniforme: True
ATENCIÓN: no se detectó un encabezado reconocible; verificar el formato antes de continuar


# Estructura oficial del CSV por estación de NOAA GHCN-D

In [8]:
COLUMNAS_NOAA = [
    "STATION",
    "DATE",
    "ELEMENT",
    "DATA_VALUE",
    "M_FLAG",
    "Q_FLAG",
    "S_FLAG",
    "OBS_TIME"
]

estructura_valida = (
    set(recuento_columnas.keys()) == {len(COLUMNAS_NOAA)}
    and not parece_encabezado
    and filas_vacias == 0
)

print("Columnas esperadas:", COLUMNAS_NOAA)
print("Cantidad esperada:", len(COLUMNAS_NOAA))
print("Cantidad observada:", dict(recuento_columnas))
print("CSV sin encabezado:", not parece_encabezado)
print("Estructura válida:", estructura_valida)

if not estructura_valida:
    raise ValueError("El archivo no cumple la estructura esperada")

Columnas esperadas: ['STATION', 'DATE', 'ELEMENT', 'DATA_VALUE', 'M_FLAG', 'Q_FLAG', 'S_FLAG', 'OBS_TIME']
Cantidad esperada: 8
Cantidad observada: {8: 47998}
CSV sin encabezado: True
Estructura válida: True


## Eliminar descarga

In [9]:
# Eliminar únicamente el CSV utilizado en la verificación
if archivo.is_file():
    archivo.unlink()
    print(f"Archivo eliminado: {archivo}")
else:
    print("El archivo ya no existe.")

Archivo eliminado: NZ000093012.csv
